In [1]:
from src.utils import create_collate_fn
from src.components.triplet_dataset import TripletDataset
from src.components.model_builder import Model
from dataclasses import dataclass
import pandas as pd
import torch.nn as nn
import torch
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

/home/cry_more/miniconda3/envs/log_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
model = Model().to('cuda')
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 13467.83it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not 

In [8]:
tokens = tokenizer.tokenize("dlk")

In [9]:
model(tokens)

TypeError: DebertaV2Model(
  (embeddings): DebertaV2Embeddings(
    (word_embeddings): Embedding(128100, 768, padding_idx=0)
    (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): DebertaV2Encoder(
    (layer): ModuleList(
      (0-5): 6 x DebertaV2Layer(
        (attention): DebertaV2Attention(
          (self): DisentangledSelfAttention(
            (query_proj): Linear(in_features=768, out_features=768, bias=True)
            (key_proj): Linear(in_features=768, out_features=768, bias=True)
            (value_proj): Linear(in_features=768, out_features=768, bias=True)
            (pos_dropout): Dropout(p=0.1, inplace=False)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): DebertaV2SelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True, bias=True)
       

In [ ]:
@dataclass
class TrainerConfig:
    model_path = "/home/cry_more/ongoing/LogRecall/artifacts"
    df_path = "/home/cry_more/ongoing/LogRecall/artifacts/stage1_triplet_data.parquet"
    batch_size=100
    max_len=128

class Trainer:
    def __init__(self,epochs=10,lr=2e-5):
        self.config = TrainerConfig()
        self.device = torch.device('cuda' if torch.cuda.is_available() else "cpu")
        self.model = Model().to(self.device)
        self.tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")
        self.epochs = epochs
        self.lr = lr
    
    def eval(self,val_dataloader):
        total_loss=0
        self.model.eval()
        with torch.no_grad():
            for batch in val_dataloader:
                
                anchor_inputs = {k: v.to(self.device) for k, v in batch["anchor"].items()}
                pos_inputs = {k: v.to(self.device) for k, v in batch["positive"].items()}
                neg_inputs = {k: v.to(self.device) for k, v in batch["negative"].items()}
                
                vec_anchor = self.model(anchor_inputs)
                vec_pos = self.model(pos_inputs)
                vec_neg = self.model(neg_inputs)

                loss = self.loss_function(vec_anchor,vec_pos,vec_neg)
                total_loss += loss.item()

        return total_loss/len(val_dataloader)

    def InitiateTraining(self):
        df = pd.read_parquet(self.config.df_path)
        X=df['masked_log']
        y=df['hash_id']
        X_train,X_val,y_train,y_val =  train_test_split(X,y,test_size=.20,random_state=33)
        X_train = X_train.reset_index(drop=True)
        y_train = y_train.reset_index(drop=True)
        X_val = X_val.reset_index(drop=True)
        y_val = y_val.reset_index(drop=True)

        train_dataset = TripletDataset(X_train,y_train)
        val_dataset = TripletDataset(X_val,y_val)

        train_dataloader = DataLoader(
            train_dataset, batch_size=self.config.batch_size, num_workers=3,
            collate_fn=create_collate_fn(self.tokenizer,self.config.max_len)
        )
        val_dataloader = DataLoader(
            val_dataset, batch_size=self.config.batch_size, num_workers=3,
            collate_fn=create_collate_fn(self.tokenizer,self.config.max_len)
        )

        self.loss_function = nn.TripletMarginWithDistanceLoss(
            distance_function=lambda x, y: 1.0 - torch.nn.functional.cosine_similarity(x, y),
            margin=0.3
        )
        self.optimizer = torch.optim.AdamW(self.model.parameters(),lr=self.lr)

        for epoch in range(self.epochs):
            self.model.train()
            total_loss=0
            for batch in train_dataloader:
                anchor_inputs = {k: v.to(self.device) for k, v in batch["anchor"].items()}
                pos_inputs = {k: v.to(self.device) for k, v in batch["positive"].items()}
                neg_inputs = {k: v.to(self.device) for k, v in batch["negative"].items()}
                
                vec_anchor = self.model(anchor_inputs)
                vec_pos = self.model(pos_inputs)
                vec_neg = self.model(neg_inputs)

                loss = self.loss_function(vec_anchor,vec_pos,vec_neg)

                self.optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()

                total_loss += loss.item()
            
            train_loss = total_loss/len(train_dataloader)
            val_loss = self.eval(val_dataloader)

            print(f"for {epoch+1} epoch train_loss-> {train_loss} and val_loss-> {val_loss}")
        
        torch.save(self.model.state_dict(), self.config.model_path + "stage1_encoder.pth")


if __name__ == "__main__":
    obj = Trainer(1)
    obj.InitiateTraining()

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 25377.80it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not 